# Assignment 4: Frontier ML: Advanced Learning Paradigms

**Student Name:** Sebastian Campos      

**Student ID:** [redacted]

**Points:** 11

## Assignment Overview

This assignment tours the advanced learning paradigms of the second half of the course. Problem 1 builds a recommender and a multilabel classifier on one anime ratings dataset. Problem 2 covers transfer learning and semi-supervised learning. Problem 3 mines structure from a network. Problem 4 simulates federated learning.

**Note:** Use your Student ID as the random seed everywhere a seed is required. Report results over **three seeds** wherever a problem says so.

**Compute note.** If your computer runs out of memory or time, reduce the sample size (fewer users or anime in Problem 1, fewer or smaller images and fewer epochs in Problem 2a, a smaller Covertype subsample in Problem 4) and still complete every part of the assignment. State exactly what you reduced, and by how much, in a short note at the top of your notebook. A smaller run that you finish and explain earns full credit for method; a run that does not finish earns none. 

## Dataset Information

| Problem | Dataset | Exact source |
|---|---|---|
| 1a, 1b | **Anime Recommendations Database** (MyAnimeList data, CooperUnion): `anime.csv` (`anime_id, name, genre, type, episodes, rating, members`) and `rating.csv` (`user_id, anime_id, rating`; -1 means watched but not scored). About 7.8 million ratings, 73,000 users, 12,000 anime. | Kaggle page: https://www.kaggle.com/datasets/CooperUnion/anime-recommendations-database (free Kaggle account; click Download, unzip to get both CSV files). A copy is also on Canvas. |
| 2a | **CIFAR-10** (Krizhevsky, 2009) | Official page: https://www.cs.toronto.edu/~kriz/cifar.html ; in code, `torchvision.datasets.CIFAR10(root="data", download=True)` |
| 2b | Breast Cancer Wisconsin (Diagnostic) | Built in: `sklearn.datasets.load_breast_cancer()` ; original source https://archive.ics.uci.edu/dataset/17 |
| 3 | Zachary's Karate Club graph | Built in: `networkx.karate_club_graph()` ; docs https://networkx.org/documentation/stable/reference/generated/networkx.generators.social.karate_club_graph.html |
| 4 | **Forest Covertype** (581,012 cells, 7 forest types, 54 features) | In code: `sklearn.datasets.fetch_covtype()` ; original source https://archive.ics.uci.edu/dataset/31/covertype |

**Loading the anime files.** Put `anime.csv` and `rating.csv` in your working folder and read them with `pd.read_csv`. Do not commit the 7.8-million-row file to GitHub; work from the subset below.

**Working subset for Problem 1 (state it in your notebook).** Keep the 2,000 most-rated anime and a random sample of 10,000 users who rated at least 20 of them; drop ratings of -1. Report the final number of users, items, and ratings, and the density.

## Import Required Libraries

Import all necessary libraries for data analysis, visualization, and machine learning.

In [1]:
# Import necessary libraries for data analysis, visualization, and machine learning
# YOUR CODE HERE:
import pandas as pd
import numpy as np
import os

---
## Problem 1: Recommender Systems & Multilabel Classification  (3 points)

### Problem 1a: Recommender Evaluated as a Ranking Task (1.5 points)

**Instructions:**

Build recommenders on the Problem 1 subset and judge them by how well they rank, not only by rating error.

* Split: hold out a random 20% of each user's ratings (seeded). A held-out rating of 8 or higher counts as *relevant*.
* Required models: (i) a **popularity** baseline, (ii) an **item-based collaborative filter** (cosine similarity), and (iii) **one latent-factor model** of your choice (implicit ALS, truncated SVD, or similar).
* Metrics: precision@10, recall@10 and NDCG@10 over all users who have at least one relevant held-out item (the same protocol as the class lab; a rating of 8 on the 10-point scale plays the role of the lab's 4 on a 5-point scale). Exclude items the user already rated in training. Show a random baseline and the three models in one table.
* Also print the top-5 recommendations (anime names) for one user of your choice.

In [ ]:
# TODO: Load rating.csv and anime.csv, build the working subset, report users/items/ratings/density
# TODO: Split 80/20 per user; define relevant = held-out rating >= 8
# TODO: Write evaluate(score_matrix) returning precision@10, recall@10 and NDCG@10
# TODO: Build popularity, item-based CF, and one latent-factor model; evaluate all three
# TODO: Print the top-5 recommended anime for one chosen user
# YOUR CODE HERE:

# Load the anime and ratings datasets
anime = pd.read_csv("animedatabase/anime.csv")
ratings = pd.read_csv("animedatabase/rating.csv")

# Filter out invalid ratings (where rating is -1)
valid_ratings = ratings[ratings['rating'] != -1]

anime_counts = valid_ratings['anime_id'].value_counts()
top_anime_ids = anime_counts.head(2000).index
sub = valid_ratings[valid_ratings['anime_id'].isin(top_anime_ids)]


user_count = sub['user_id'].value_counts()
eligible_users = user_count[user_count >= 20].index

userseed = int(os.environ["STUDENT_ID"])
randomNumberGenerator = np.random.default_rng(seed=userseed)
sampled_users = randomNumberGenerator.choice(eligible_users, size=10000, replace=False)

n_user = sub[sub['user_id'].isin(sampled_users)]['user_id'].nunique()
n_item = sub[sub['user_id'].isin(sampled_users)]['anime_id'].nunique()
n_rating = sub[sub['user_id'].isin(sampled_users)].shape[0]         
density = n_rating / (n_user * n_item)

print(f"Users: {n_user:,}  Items: {n_item:,}  Ratings: {n_rating:,}  Density: {density:.4f}")

# Finalize the working subset from your sampled users
working_subset = sub[sub['user_id'].isin(sampled_users)].copy()

# Shuffle using your seed to randomize each user's interaction order
shuffled_subset = working_subset.sample(frac=1.0, random_state=userseed).reset_index(drop=True)

# Assign a normalized position (0.0 to 1.0) within each user's interactions
shuffled_subset['user_rank'] = shuffled_subset.groupby('user_id').cumcount()
shuffled_subset['user_total'] = shuffled_subset.groupby('user_id')['anime_id'].transform('count')
shuffled_subset['split_ratio'] = shuffled_subset['user_rank'] / shuffled_subset['user_total']

# Split 80% into train and 20% into test per user
train_df = shuffled_subset[shuffled_subset['split_ratio'] < 0.8].drop(columns=['user_rank', 'user_total', 'split_ratio'])
test_df = shuffled_subset[shuffled_subset['split_ratio'] >= 0.8].drop(columns=['user_rank', 'user_total', 'split_ratio'])


# Define relevant items in the held-out set (rating >= 8)
test_df['relevant'] = (test_df['rating'] >= 8).astype(int)

# Map each user_id to their set of relevant held-out anime_ids
ground_truth_relevant = (
    test_df[test_df['relevant'] == 1]
    .groupby('user_id')['anime_id']
    .apply(set)
    .to_dict()
)

scores = np.array([
    [0.9, 0.1, 0.8, 0.3, 0.7, 0.2, 0.6, 0.4],   # user 0
    [0.2, 0.9, 0.1, 0.8, 0.3, 0.7, 0.4, 0.6],   # user 1
    [0.5, 0.4, 0.3, 0.2, 0.9, 0.8, 0.1, 0.7],   # user 2
])

train_items = {0: {0}, 1: {1,3},2: set()}
relevant    = {0: {2, 6}, 1: {5}, 2: {4, 7, 1}}     # relevant HELD-OUT items (columns)

K = 3

precisions , recalls , ndcg_scores = [],[],[]


for u, rel in relevant.items():
    row = scores[u].copy()                       # copy so the original isn't changed
    row[list(train_items[u])] = -np.inf          # exclude items already rated in training

    top_k = np.argsort(-row)[:K]                 # columns of the K highest scores, best first
    hits = [1 if item in rel else 0 for item in top_k]

    precisions.append(sum(hits) / K)
    recalls.append(sum(hits) / len(rel))

    dcg  = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), K)))
    ndcg_scores.append(dcg / idcg)

    print(f"user {u}: top-{K} = {top_k.tolist()}, hits = {hits}")

print(f"\nP@{K}: {np.mean(precisions):.3f}  R@{K}: {np.mean(recalls):.3f}  NDCG@{K}: {np.mean(ndcg_scores):.3f}")

Users: 10,000  Items: 2,000  Ratings: 1,150,568  Density: 0.0575


### 📝 Analysis Explanation (1a)

**Instructions:** Which model ranks best, and does the ordering match what you would expect from the class lab? Explain why the popularity baseline can be hard to beat. What is the cold-start problem, and how would it affect your best model for a brand-new anime or a brand-new user?

**Your Analysis:** [Write your interpretation here.]

### Problem 1b: Multilabel Genre Classification (1.5 points)

**Instructions:**

Each anime lists several genres in the `genre` column (comma-separated), so genre prediction is a multilabel problem. Keep the 12 most frequent genres as labels and the anime that have at least one of them.

* Features, built per anime: `type` (one-hot), `episodes`, `members`, a TF-IDF of the `name`, and **audience features** (a 30-component truncated SVD of the anime-by-user matrix (as in the class lab) from your training ratings).
* Models: a trivial baseline (always predict the most common genre), **binary relevance** (one logistic regression per genre), and a **classifier chain**.
* Evaluate with Hamming loss, subset accuracy, micro F1, and macro F1 on a held-out 25% of anime. Compare feature sets: name text only, audience only, all features.

In [3]:
# TODO: Build the 12-genre label matrix and the three feature sets (text, audience SVD, all)
# TODO: Train the always-most-common baseline, binary relevance, and a classifier chain
# TODO: Report Hamming loss, subset accuracy, micro F1, macro F1 for each model and feature set in one table
# TODO: Print per-genre F1 for your best model
# YOUR CODE HERE:

### 📝 Analysis Explanation (1b)

**Instructions:** Why is plain accuracy a poor metric here, and which metric would have flattered your baseline? Which feature set carries more genre signal, and why? Which genres does your best model predict well, and which does it struggle with? Did the classifier chain help?

**Your Analysis:** [Write your interpretation here.]

---
## Problem 2: Transfer Learning & Semi-Supervised Learning  (2 points)

### Problem 2a: Transfer Learning with a Learning Curve

**Instructions:**

Compare three strategies on CIFAR-10 for several amounts of labeled data: (i) a small CNN or ResNet18 trained **from scratch**, (ii) a **pretrained ResNet18 with a frozen encoder** and a new classification layer (feature extraction), and (iii) the pretrained ResNet18 **fine-tuned** with a small learning rate. Use 100, 500, and 2,000 labeled training images (class-balanced), evaluate on the same 2,000 held-out test images, and repeat each setting over 3 seeds.

In [4]:
# TODO: Load CIFAR-10; build balanced labeled subsets of 100, 500, and 2000 images; fix a 2000-image test set
# TODO: Train from scratch, frozen-encoder + new head, and fine-tuned ResNet18 for each size and seed (a few epochs each)
# TODO: Record test accuracy; compute mean and standard deviation over the 3 seeds
# TODO: Plot accuracy against number of labeled images, one line per strategy, with error bars
# YOUR CODE HERE:

### 📝 Analysis Explanation (2a)

**Instructions:** At which label budget does transfer help most, and when does the gap narrow? When would you choose feature extraction over full fine-tuning? Why is repeating over seeds necessary here?

**Your Analysis:** [Write your interpretation here.]

### Problem 2b: Semi-Supervised Learning

**Instructions:**

Using the Breast Cancer Wisconsin dataset, hide 80% of the training labels (treat them as unlabeled, label -1). Apply self-training: fit on the labeled 20%, add high-confidence pseudo-labels, and repeat. Compare (i) a model trained only on the 20% labeled data, (ii) the self-trained model at two confidence thresholds (for example 0.7 and 0.95), and (iii) an oracle trained on all labels. Repeat over 5 random splits and report mean and standard deviation.

In [5]:
# TODO: Load load_breast_cancer(), split into train/test, hide 80% of train labels (-1)
# TODO: Fit the labeled-only baseline, SelfTrainingClassifier at two thresholds, and the all-labels oracle
# TODO: Repeat over 5 random splits; report mean and std of test accuracy in one table
# TODO: Report how many pseudo-labels were added and what share of them were correct (you know the true labels)
# YOUR CODE HERE:

### 📝 Analysis Explanation (2b)

**Instructions:** Did self-training beat the labeled-only baseline, and by how much compared with the spread across splits? Use the share of correct pseudo-labels to explain the result. Under what conditions would self-training hurt?

**Your Analysis:** [Write your interpretation here.]

---
## Problem 3: Graph & Network Mining  (2 points)

### Problem 3: Graph & Network Mining

**Instructions:**

Load Zachary's Karate Club graph. Compute degree centrality, betweenness centrality, and PageRank; detect communities with Louvain (`networkx.community.louvain_communities`) and compare them with the known club split stored in the node attribute `club` (use normalized mutual information) and report the modularity of the Louvain partition. Compare the top-5 nodes by PageRank with the top-5 by degree and by betweenness. Then run a small **link-prediction** test: hide 20% of the edges, score the hidden edges and random non-edges with common neighbors and the Adamic-Adar index, and report ROC AUC for both. Visualize the graph colored by community.

In [6]:
# TODO: Load nx.karate_club_graph(); compute degree, betweenness, and PageRank for every node
# TODO: Run Louvain; compute NMI against the node attribute 'club'; plot the graph colored by community
# TODO: Hide 20% of the edges, score with the Adamic-Adar index, and compute ROC AUC against sampled non-edges (repeat over several random hides)
# YOUR CODE HERE:

### 📝 Analysis Explanation (3)

**Instructions:** Which nodes are most central by each measure, and why might the three measures disagree? What do the detected communities represent, and how closely do they match the real split? Is the link-prediction AUC trustworthy on a graph this small?

**Your Analysis:** [Write your interpretation here.]

---
## Problem 4: Federated Learning  (2 points)

### Problem 4: Federated Learning

**Instructions:**

Use the Forest Covertype data (take a random 60,000-row subsample; scale the features). Define **four clients by wilderness area** (the four one-hot columns 10 to 13), so each client has a different class mix. Train a logistic regression locally on each client, then average the coefficients and intercepts (weighted by client size) into one federated model. Compare test accuracy of (i) the federated average, (ii) a model trained on the pooled data, and (iii) the average accuracy of the individual local models, each evaluated on a test set drawn from all four areas. Then repeat the comparison with an **IID split** (four random clients of equal size) as a control.

In [7]:
# TODO: Fetch Covertype; take a random 60,000-row subsample; scale features; hold out a test set from all areas
# TODO: Build four clients by wilderness area; train one logistic regression per client
# TODO: Average coefficients and intercepts weighted by client size; evaluate the federated model
# TODO: Train the pooled model; evaluate it and the individual local models
# TODO: Repeat with four random (IID) clients and report both settings in one table
# YOUR CODE HERE:

### 📝 Analysis Explanation (4)

**Instructions:** How much accuracy did federating cost in the non-IID split compared with the IID split? Why does non-IID data hurt a single round of averaging? Federated learning trades some accuracy for what practical benefit, and in what real-world scenario would you accept that trade?

**Your Analysis:** [Write your interpretation here.]

---
## Results Interpretation  (2 points)

**Instructions:** Step back from the individual sub-parts and write a short, integrated interpretation of this assignment's results. Address: (1) which of the paradigms you explored seems most directly applicable to a project you might actually pursue; (2) one real constraint (data, compute, or privacy) that would push you toward one of these paradigms over standard supervised learning; and (3) which paradigm you found hardest to get working well, and your best guess as to why.

Write 150-250 words. This section is graded on the quality of your reasoning and how well you tie your specific numbers and figures back into a clear conclusion, not on reaching any particular "correct" answer.

**Your Interpretation:** [Write your response here.]

---
## AI Usage Statement (required)

Briefly state which AI tools you used (if any) and for what, consistent with the course AI Tool Policy.

> [Example: "GitHub Copilot was used for boilerplate plotting code. All analysis and interpretation are my own work."]